## Mini Project: E-Commerce Orders Data Validation Pipeline

Objective

- Build a PySpark pipeline that ingests raw order data, identifies bad records, applies basic transformations, and produces a clean output dataset.

- You should complete this project using only the topics you've learned so far:
    - Read Files With PySpark
    - Handle Malformed Records
    - Define Schema
    - Select
    - Alias
    - Filter

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

from pyspark.sql.functions import col

### Task 1 — Define Schema Explicitly for orders.csv

In [0]:
orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("city", StringType(), True),
    StructField("amount", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])

### Task 2 — Read Raw Orders File

In [0]:
raw_orders_df = spark.read.format("csv")\
                .option("header", True)\
                .option("mode", "PERMISSIVE")\
                .option("columnNameOfCorruptRecord", "_corrupt_record")\
                .schema(orders_schema)\
                .load("/Volumes/workspace/learning_spark/spark/mini_project_videos_35_41/orders.csv")

display(raw_orders_df)

### Task 3 — Handle Malformed Records

In [0]:
bad_records_df = raw_orders_df.filter(
    col("_corrupt_record").isNotNull()
)

display(bad_records_df)

In [0]:
good_records_df = raw_orders_df.filter(
    col("_corrupt_record").isNull()
).drop("_corrupt_record")

display(good_records_df)

### Task 4 — Read Product Master File

In [0]:
products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True)
])

products_df = spark.read.format("csv")\
            .option("header", True)\
            .schema(products_schema)\
            .load("/Volumes/workspace/learning_spark/spark/mini_project_videos_35_41/products.csv")
        
display(products_df)

### Task 5 — Select Required Columns and Rename Using Alias

From orders data keep only:

- order_id (rename to order_number)
- customer_id (rename to customer_number)
- product_id
- city
- amount (rename to order_amount)
- status

Remove:

- order_date

Output:

- renamed_orders_df

In [0]:
renamed_orders_df = good_records_df.select(
    col("order_id").alias("order_number"),
    col("customer_id").alias("customer_number"),
    col("product_id"),
    col("city"),
    col("amount").alias("order_amount"),
    col("status")
)

display(renamed_orders_df)

### Task 6 — Filter Completed Orders

Keep only:

- status = COMPLETED

Output:

- completed_orders_df

In [0]:
completed_orders_df = renamed_orders_df.filter(
    col("status") == "COMPLETED"
)

display(completed_orders_df)

### Task 7 — Filter High-Value Orders

From completed orders keep only:

- order_amount > 2000

Output:

- high_value_orders_df

In [0]:
high_value_orders_df = completed_orders_df.filter(
    col("order_amount") > 2000
)

display(high_value_orders_df)

### Task 8 — Filter Target Cities

Keep orders only from:

- Pune
- Mumbai
- Hyderabad

Output:

- target_city_orders_df

In [0]:
target_cities = ["Pune", "Mumbai", "Hyderabad"]

target_city_orders_df = high_value_orders_df.filter(
    col("city").isin(target_cities)
)

display(target_city_orders_df)

### Task 9 — Identify Suspicious Business Records

Create a separate DataFrame containing:

- Negative amounts

    Example: 1011,C111,P011,Mumbai,-500,COMPLETED,2026-06-01

- Extremely large amounts (Treat anything above ₹50,000 as suspicious)

    Example: 1016,C116,P016,Mumbai,999999,COMPLETED,2026-06-01

Output:

- suspicious_orders_df

In [0]:
suspicious_orders_df = renamed_orders_df.filter(
    (col("order_amount") < 0) |
    (col("order_amount") > 50000)
)

display(suspicious_orders_df)

### Task 10 — Identify Records With Missing Values

Examples:

1010,C110,P010,Pune,,COMPLETED,2026-06-01

1013,C113,P013,,2100,COMPLETED,2026-06-01

Output:

- missing_value_orders_df

![image_1781778489687.png](./image_1781778489687.png "image_1781778489687.png")

In [0]:
missing_value_orders_df = renamed_orders_df.filter(
    col("order_number").isNull() |
    col("customer_number").isNull() |
    col("product_id").isNull() |
    col("city").isNull() |
    col("order_amount").isNull() |
    col("status").isNull()
)

display(missing_value_orders_df)

### Task 11 — Create Final Clean Dataset

A record is considered clean if:

- Not malformed
- Status = COMPLETED
- Amount > 2000
- City is Pune, Mumbai, or Hyderabad
- Amount is not negative
- Amount is not suspiciously large

Output:

- final_orders_df

In [0]:
final_orders_df = completed_orders_df.filter(
    (col("city").isin(target_cities)) &
    (col("order_amount") > 2000) &
    (col("order_amount") <= 50000)
)

display(final_orders_df)